[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/02-o-que-e-ml/notebooks/02_demo.ipynb)

# Aula 2 — Demonstração: do problema ao primeiro modelo
*Computação Aplicada à Economia: Tópicos em Aprendizado de Máquina*

Vamos percorrer o fluxo completo de um projeto com uma base clássica de preços de imóveis:

**pergunta → dados → separar → treinar → avaliar → comunicar**

> A base é americana (censo da Califórnia, 1990). Usamos porque é limpa e pública; o objetivo hoje é aprender o *fluxo*.

## 1. A pergunta (ficha de formulação)

| Item | Resposta |
|---|---|
| **Decisão** | precificar imóveis em regiões sem transações recentes |
| **Unidade** | um quarteirão censitário |
| **Alvo y** | valor mediano dos imóveis (em US$ 100 mil) |
| **Atributos x** | renda mediana, idade dos imóveis, cômodos, população, localização |
| **Tarefa** | regressão |
| **Métrica** | erro típico da previsão (RMSE), na mesma unidade de y |

## 2. Os dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.datasets import fetch_california_housing
    df = fetch_california_housing(as_frame=True).frame
    print("Base carregada:", df.shape)
except Exception as erro:  # sem internet: base sintética com a mesma estrutura
    print("Usando base simulada:", erro)
    rng = np.random.default_rng(0)
    n = 5000
    df = pd.DataFrame({
        "MedInc": rng.lognormal(1.2, 0.45, n), "HouseAge": rng.integers(1, 52, n),
        "AveRooms": rng.normal(5.4, 1.2, n), "AveBedrms": rng.normal(1.1, 0.2, n),
        "Population": rng.lognormal(7, 0.7, n), "AveOccup": rng.normal(3, 0.8, n),
        "Latitude": rng.uniform(32.5, 42, n), "Longitude": rng.uniform(-124.3, -114.3, n)})
    df["MedHouseVal"] = (0.45 * df.MedInc + 0.01 * df.HouseAge
                         - 0.1 * (df.Latitude - 34) + rng.normal(0, 0.6, n)).clip(0.15, 5)

df.head()

In [ ]:
# Cada linha é uma observação (quarteirão); cada coluna, uma variável
alvo = "MedHouseVal"
y = df[alvo]
X = df.drop(columns=alvo)
print(f"n = {X.shape[0]} observações, p = {X.shape[1]} atributos")
X.describe().T[["mean", "std", "min", "max"]]

## 3. Separar: o teste fica guardado na gaveta

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
len(X_tr), len(X_te)

## 4–5. Treinar e avaliar (sempre contra uma referência ingênua)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def rmse(modelo, X, y):
    return mean_squared_error(y, modelo.predict(X)) ** 0.5

resultados = {}
for nome, modelo in [("média (ingênuo)", DummyRegressor(strategy="mean")),
                     ("regressão linear", LinearRegression())]:
    modelo.fit(X_tr, y_tr)                       # aprende SÓ com o treino
    resultados[nome] = rmse(modelo, X_te, y_te)  # avalia no teste
pd.Series(resultados, name="RMSE no teste (US$ 100 mil)").round(3)

### Trocar o modelo é trocar uma linha

In [ ]:
from sklearn.ensemble import RandomForestRegressor

floresta = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0)
floresta.fit(X_tr, y_tr)
resultados["floresta aleatória"] = rmse(floresta, X_te, y_te)
pd.Series(resultados, name="RMSE no teste").round(3)

### Teaser da Aula 3: aprendeu ou decorou?
Compare o erro **no treino** com o erro **no teste**.

In [ ]:
pd.DataFrame({
    "treino": [rmse(m, X_tr, y_tr) for m in (modelo, floresta)],
    "teste":  [rmse(m, X_te, y_te) for m in (modelo, floresta)],
}, index=["regressão linear", "floresta aleatória"]).round(3)

### Cuidado ao interpretar coeficientes
Os coeficientes abaixo ajudam a **prever**. Eles **não** são efeitos causais: um cômodo a mais não "causa" a variação indicada no preço.

In [ ]:
pd.Series(modelo.coef_, index=X.columns).round(3).sort_values()

## Bônus: aprendizado não supervisionado — regiões a partir das coordenadas

In [ ]:
from sklearn.cluster import KMeans

coords = df[["Longitude", "Latitude"]]
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(coords)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(coords.Longitude, coords.Latitude, c=km.labels_, s=2, cmap="tab10")
ax.scatter(*km.cluster_centers_.T, marker="X", s=150, c="black")
ax.set(xlabel="longitude", ylabel="latitude", title="5 grupos propostos pelo k-médias")
plt.show()

Ninguém disse ao algoritmo onde ficam Los Angeles ou São Francisco. Ele propôs grupos de pontos próximos; **nós** interpretamos.